# Proyek Sentimen Analisis

** Sumber Data: Awal KEHANCURAN Bank Mandiri ?!
** Link Video: https://youtu.be/d-r2cVlFSwg?si=aQyfB70XfujMtRX6

**Nama Proyek: Mandiri PR Crisis & Risk Monitor (Sentiment Analysis System)

**Fokus Proyek: Mitigasi Risiko Reputasi & Analisis Sentimen menggunakan Machine Learning / Deep Learning.

## Install dependencies, Imports, configuration, API key

In [16]:
!pip install -q google-api-python-client pandas numpy scikit-learn Sastrawi nltk \
    transformers peft accelerate torch joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 11.4 MB/s eta 0:00:00


In [17]:
import os, re, html, time, random, warnings
from functools import lru_cache

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------------- Global configuration ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

VIDEO_ID   = "d-r2cVlFSwg"      # Raymond Chin video (PRD Section 3)
MIN_ROWS   = 3000               # PRD: minimum 3,000 samples
MAX_ROWS   = 10000              # safety cap for scraping
RAW_CSV    = "raw_comments.csv"
CLEAN_CSV  = "clean_dataset.csv"

LABEL2ID = {"Negatif": 0, "Netral": 1, "Positif": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = [ID2LABEL[i] for i in range(3)]

In [2]:
# ---------------- API key ----------------
# Preferred: Colab Secrets (key name: YOUTUBE_API_KEY). Fallbacks: env var, then prompt.
try:
    from google.colab import userdata
    API_KEY = userdata.get("YOUTUBE_API_KEY")
except Exception:
    from getpass import getpass
    API_KEY = os.environ.get("YOUTUBE_API_KEY") or getpass("YouTube Data API v3 key: ")

YouTube Data API v3 key: ··········


## Scraping function (YouTube Data API v3 with pagination)

In [3]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def _execute(request, retries=3):
    """Execute an API request with simple exponential back-off on transient 5xx errors."""
    for attempt in range(retries):
        try:
            return request.execute()
        except HttpError as e:
            if e.resp.status in (500, 502, 503, 504) and attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise


def _to_row(comment: dict, parent_id: str = None) -> dict:
    """Map an API comment resource to the attributes required by the PRD."""
    s = comment["snippet"]
    return {
        "Comment_ID":    comment["id"],
        "Author_Name":   s.get("authorDisplayName"),
        "Text_Original": s.get("textDisplay", ""),
        "Like_Count":    s.get("likeCount", 0),
        "Published_At":  s.get("publishedAt"),          # UTC (ISO 8601)
        "Parent_ID":     parent_id,                     # extra: None for top-level comments
    }


def _fetch_all_replies(youtube, parent_id: str) -> list:
    """commentThreads only embeds a few replies, so page through ALL replies of a thread."""
    out, token = [], None
    while True:
        resp = _execute(youtube.comments().list(
            part="snippet", parentId=parent_id, maxResults=100,
            pageToken=token, textFormat="plainText"))
        out += [_to_row(c, parent_id) for c in resp.get("items", [])]
        token = resp.get("nextPageToken")
        if not token:
            return out


def scrape_youtube_comments(video_id: str, api_key: str, min_rows: int = 3000,
                            max_rows: int = 10000) -> pd.DataFrame:
    """Scrape top-level comments + replies with full pagination."""
    youtube = build("youtube", "v3", developerKey=api_key)
    rows = {}  # Comment_ID -> row (deduplicates across passes)

    for order in ("time", "relevance"):          # 2nd pass only if the 1st is not enough
        token = None
        try:
            while len(rows) < max_rows:
                resp = _execute(youtube.commentThreads().list(
                    part="snippet,replies", videoId=video_id, maxResults=100,
                    pageToken=token, order=order, textFormat="plainText"))

                for item in resp.get("items", []):
                    top = item["snippet"]["topLevelComment"]
                    rows[top["id"]] = _to_row(top)

                    n_replies = item["snippet"].get("totalReplyCount", 0)
                    inline = item.get("replies", {}).get("comments", [])
                    if n_replies > len(inline):
                        replies = _fetch_all_replies(youtube, top["id"])
                    else:
                        replies = [_to_row(c, top["id"]) for c in inline]
                    for r in replies:
                        rows[r["Comment_ID"]] = r

                token = resp.get("nextPageToken")
                print(f"[{order}] collected: {len(rows):,}", end="\r")
                if not token:
                    break
        except HttpError as e:
            print(f"\nAPI error ({e.resp.status}): {e}. Returning what was collected so far.")
        if len(rows) >= min_rows:
            break

    df = pd.DataFrame(rows.values())
    df = df[df["Text_Original"].fillna("").str.strip() != ""].reset_index(drop=True)
    print(f"\nTotal scraped rows: {len(df):,}")
    if len(df) < min_rows:
        print(f"WARNING: only {len(df)} rows (< {min_rows}). The video may simply have fewer comments.")
    return df

In [5]:
# Run scraping and save raw_comments.csv
raw_df = scrape_youtube_comments(VIDEO_ID, API_KEY, min_rows=MIN_ROWS, max_rows=MAX_ROWS)
raw_df.to_csv(RAW_CSV, index=False, encoding="utf-8")
raw_df.head()

[time] collected: 4,599
Total scraped rows: 4,599


,Comment_ID,Author_Name,Text_Original,Like_Count,Published_At,Parent_ID
0,UgwEm_yYEU6REGjuAgB4AaABAg,@nandaperdana8264,AYO BOIKOT MANDIRI!!!,0,2026-10-10T05:06:44Z,None
1,UgxzI2O0aEzYlbKtiER4AaABAg,@yafaliahalawa6849,Ka uang kami juga Hilang bang di mandiri 7 jut...,0,2026-10-07T07:56:38Z,None
2,UgzH8e8zDpBmTq6oR5N4AaABAg,@tatangsutisna3786,Bank Mandiri harus membayar mahal dengan menju...,0,2026-10-07T02:14:16Z,None
3,UgwSwkFhOHKZnRBtOVp4AaABAg,@nurirfan1997,"Mandri emang gk jelas tiba"" akun livin gk bisa...",0,2026-10-02T03:41:00Z,None
4,UgzDCw4TWKmlLM1WlMF4AaABAg,@rahmalarasati9212,i love u bang,0,2026-09-28T22:53:55Z,None


## NLP pre-processing pipeline (the 8 PRD steps)

In [6]:
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# =========================================================
# STEP 1: Noise removal (URL, @mention, #hashtag, digits, punctuation)
# Emojis are intentionally preserved here; they are converted in Step 2.
# =========================================================
URL_RE     = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
DIGIT_RE   = re.compile(r"\d+")
PUNCT_RE   = re.compile(r"[^\w\s\U0001F300-\U0001FAFF\u2600-\u27BF]")

def noise_removal(text: str) -> str:
    text = html.unescape(str(text))
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = DIGIT_RE.sub(" ", text)
    text = text.replace("_", " ")
    return PUNCT_RE.sub(" ", text)

# =========================================================
# STEP 2: Text normalization (emoji -> words, slang -> formal, word elongation)
# =========================================================
EMOJI_PATTERN = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\uFE0F\u200D]")
EMOJI_MAP = {
    "😡": "marah", "🤬": "marah", "😠": "marah", "😤": "marah",
    "😭": "sedih", "😢": "sedih", "😞": "kecewa", "😔": "kecewa",
    "😂": "lucu", "🤣": "lucu", "🤡": "badut",
    "👍": "bagus", "👏": "bagus", "🙏": "terima kasih", "❤": "suka",
    "😍": "suka", "😊": "senang", "🔥": "mantap", "💪": "semangat",
    "👎": "buruk", "🤔": "bingung", "😱": "kaget", "🤮": "muak",
}
ELONGATION_RE = re.compile(r"(.)\1{2,}")   # 3+ repeated chars -> 1  ("hancurrrr" -> "hancur")

LEXICON_URL = ("https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/"
               "colloquial-indonesian-lexicon.csv")

# Custom dictionary (netizen + banking terms). Overrides the GitHub lexicon.
CUSTOM_SLANG = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak",
    "engga": "tidak", "enggak": "tidak", "kagak": "tidak", "tdk": "tidak", "tak": "tidak",
    "bkn": "bukan", "jgn": "jangan", "blm": "belum", "udh": "sudah", "udah": "sudah",
    "sdh": "sudah", "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena",
    "karna": "karena", "jg": "juga", "aja": "saja", "bgt": "sangat", "banget": "sangat",
    "klo": "kalau", "kalo": "kalau", "emg": "memang", "dr": "dari", "sm": "sama",
    "trs": "terus", "org": "orang", "gue": "saya", "gw": "saya", "lu": "kamu",
    "rek": "rekening", "rekng": "rekening", "tf": "transfer", "wd": "tarik dana",
    "bnk": "bank", "nasabah2": "nasabah", "pake": "pakai", "bikin": "membuat",
}

def load_slang_dict() -> dict:
    """Load the Colloquial Indonesian Lexicon (GitHub); fall back to the custom dict only."""
    slang = {}
    try:
        lex = pd.read_csv(LEXICON_URL)
        slang = {str(s).lower(): str(f).lower() for s, f in zip(lex["slang"], lex["formal"])
                 if str(s).lower() != str(f).lower()}
        print(f"Loaded {len(slang):,} entries from Colloquial Indonesian Lexicon.")
    except Exception as e:
        print(f"Could not load lexicon ({e}); using custom dictionary only.")
    slang.update(CUSTOM_SLANG)
    return slang

SLANG = load_slang_dict()

def normalize_text(text: str) -> str:
    text = EMOJI_PATTERN.sub(lambda m: f" {EMOJI_MAP.get(m.group(), '')} ", text)  # unmapped emoji dropped
    text = ELONGATION_RE.sub(r"\1", text)
    return " ".join(SLANG.get(w.lower(), w) for w in text.split())

# STEP 3: Case folding
def case_folding(text: str) -> str:
    return text.lower()

# STEP 4: Spacing cleansing
def spacing_cleansing(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

# STEP 5: Tokenization
def tokenize(text: str) -> list:
    return text.split()

# =========================================================
# STEP 6: Stopword removal WITH negation handling
# =========================================================
NEGATION_WORDS = {"tidak", "bukan", "jangan", "kurang", "belum"}   # must survive

STOPWORDS = (set(StopWordRemoverFactory().get_stop_words())
             | set(nltk_stopwords.words("indonesian"))) - NEGATION_WORDS
assert NEGATION_WORDS.isdisjoint(STOPWORDS), "Negation words leaked into the stopword list!"

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in STOPWORDS]

# =========================================================
# STEP 7: Stemming (Sastrawi), negation words are protected from stemming
# =========================================================
_stemmer = StemmerFactory().create_stemmer()

@lru_cache(maxsize=None)
def stem_word(word: str) -> str:
    return word if word in NEGATION_WORDS else _stemmer.stem(word)

def stem_tokens(tokens: list) -> list:
    return [stem_word(t) for t in tokens]

# ---------------- Composite helpers ----------------
def preprocess_to_normalized(raw_text: str) -> str:
    """Steps 1-4 -> readable normalized text (also the natural input for BERT)."""
    t = noise_removal(raw_text)
    t = normalize_text(t)
    t = case_folding(t)
    return spacing_cleansing(t)

def preprocess_to_clean(normalized_text: str) -> str:
    """Steps 5-7 -> tokens, stopwords removed (negations kept), stemmed."""
    tokens = stem_tokens(remove_stopwords(tokenize(normalized_text)))
    return " ".join(tokens)

def full_pipeline(raw_text: str) -> dict:
    norm = preprocess_to_normalized(raw_text)
    return {"text_normalized": norm, "text_clean": preprocess_to_clean(norm)}

# Sanity check on a negation example
print(full_pipeline("Bank ini TIDAK aman!!! Saya kecewaaaa banget, mau tarik dana @RaymondChins"))

Loaded 4,331 entries from Colloquial Indonesian Lexicon.
{'text_normalized': 'bank ini tidak aman saya kecewa sangat mau tarik dana', 'text_clean': 'bank tidak aman kecewa tarik dana'}


In [8]:
# Rule-based labeling (proxy labels)
# Lexicons are written as plain words, then STEMMED with the same Sastrawi stemmer,
# so they match the stemmed `text_clean` tokens ("mengecewakan" -> "kecewa").
# Weight 2 = strong signal (crisis keywords), 1 = mild signal.
NEG_RAW = {
    # strong: anger, bank-run intent, accusations
    "kecewa": 2, "marah": 2, "tarik": 2, "boikot": 2, "bubar": 2, "bangkrut": 2,
    "hancur": 2, "kabur": 2, "pindah": 2, "bohong": 2, "tipu": 2, "zalim": 2,
    "arogan": 2, "sombong": 2, "rampok": 2, "maling": 2, "muak": 2, "benci": 2,
    "bodoh": 2, "goblok": 2, "tolol": 2, "sampah": 2, "bobrok": 2, "kapok": 2,
    # mild: criticism of communication / handling
    "buruk": 1, "jelek": 1, "rugi": 1, "gagal": 1, "parah": 1, "malu": 1,
    "takut": 1, "khawatir": 1, "panik": 1, "kacau": 1, "aneh": 1, "curang": 1,
    "korup": 1, "blunder": 1, "sewenang": 1, "semena": 1, "defensif": 1,
    "lambat": 1, "ribet": 1, "payah": 1, "licik": 1, "ngawur": 1, "ngaco": 1,
    "trauma": 1, "sesat": 1,
}
POS_RAW = {
    # strong: explicit support / defending the bank
    "dukung": 2, "setuju": 2, "bela": 2, "sepakat": 2,
    # mild
    "bagus": 1, "tepat": 1, "wajar": 1, "adil": 1, "sah": 1, "legal": 1,
    "profesional": 1, "paham": 1, "mantap": 1, "keren": 1, "salut": 1, "hormat": 1,
    "bijak": 1, "amanah": 1, "percaya": 1, "aman": 1, "sabar": 1, "semangat": 1,
    "sukses": 1, "apresiasi": 1, "patuh": 1, "taat": 1, "prosedur": 1, "sesuai": 1,
    "jelas": 1, "transparan": 1, "solid": 1, "makasih": 1, "positif": 1, "jaya": 1,
}
QUESTION_WORDS = {"apa", "apakah", "bagaimana", "gimana", "kenapa", "mengapa", "siapa",
                  "kapan", "dimana", "berapa", "kah"}

def _build_lex(raw: dict) -> dict:
    lex = {}
    for w, wt in raw.items():
        s = stem_word(w)
        lex[s] = max(lex.get(s, 0), wt)
    return lex

NEG_LEX, POS_LEX = _build_lex(NEG_RAW), _build_lex(POS_RAW)

def label_comment(raw_text: str, text_normalized: str, text_clean: str) -> int:
    """
    Returns 0 (Negatif), 1 (Netral), 2 (Positif).
    - Weighted lexicon score on stemmed tokens
    - Negation flip: a negation word within the 2 preceding tokens inverts the polarity
    - Questions / lurkers stay Netral unless the sentiment signal is strong (|score| >= 2)
    """
    tokens, score = text_clean.split(), 0
    for i, tok in enumerate(tokens):
        if tok in NEG_LEX:   s = -NEG_LEX[tok]
        elif tok in POS_LEX: s = POS_LEX[tok]
        else:                continue
        if any(t in NEGATION_WORDS for t in tokens[max(0, i - 2):i]):
            s = -s
        score += s

    is_question = ("?" in str(raw_text)) or any(w in QUESTION_WORDS for w in text_normalized.split()[:3])
    if is_question and abs(score) < 2:
        return 1
    if score <= -1: return 0
    if score >= 1:  return 2
    return 1

In [9]:
# Apply pipeline, Step 8 row validation, labeling, save clean_dataset.csv

raw_df = pd.read_csv(RAW_CSV)
df = raw_df[["Comment_ID", "Author_Name", "Text_Original", "Like_Count", "Published_At"]].copy()
df["Text_Original"] = df["Text_Original"].astype(str)

# Steps 1-7
df["text_normalized"] = df["Text_Original"].apply(preprocess_to_normalized)
df["text_clean"]      = df["text_normalized"].apply(preprocess_to_clean)

# STEP 8: Row validation (empty rows + duplicates/spam)
n0 = len(df)
df = df[(df["text_normalized"].str.strip() != "") & (df["text_clean"].str.strip() != "")]
df = df.dropna(subset=["text_clean"])
df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)
print(f"Row validation: {n0:,} -> {len(df):,} rows")
if len(df) < MIN_ROWS:
    print(f"WARNING: {len(df)} rows < {MIN_ROWS}. Scrape more (e.g. other pass/orders) before submission.")

# Labeling
df["label"] = [label_comment(r, n, c) for r, n, c in
               zip(df["Text_Original"], df["text_normalized"], df["text_clean"])]
df["sentiment"] = df["label"].map(ID2LABEL)

print("\nClass distribution:")
print(df["sentiment"].value_counts())
print((df["sentiment"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

# Save outputs
df.to_csv(CLEAN_CSV, index=False, encoding="utf-8")
df.to_csv("dataset.csv", index=False, encoding="utf-8")       # PRD deliverable name
df.sample(min(150, len(df)), random_state=SEED).to_csv("label_audit_sample.csv", index=False)
df[["Text_Original", "text_clean", "sentiment"]].sample(8, random_state=SEED)

Row validation: 4,599 -> 4,359 rows

Class distribution:
sentiment
Netral     2525
Negatif    1215
Positif     619
Name: count, dtype: int64
sentiment
Netral     57.9 %
Negatif    27.9 %
Positif    14.2 %
Name: proportion, dtype: object


,Text_Original,text_clean,sentiment
1509,Sudah ga aman kawan kawan nabung dibank mandir...,tidak aman kawan kawan nabung bank mandiri baw...,Netral
2232,"Pemerintah bermain-main dengan TRUST, lalu men...",perintah main main trust sebar aparat polisi s...,Negatif
643,Pindah bank biar jadi pelajaran buat bank” lain,pindah bank biar ajar bank,Negatif
2346,Bank Mandiri Bangkrut,bank mandiri bangkrut,Negatif
3917,Semua cabang saya selama ini memakai tabungan ...,cabang pakai tabung mandiri gaji gaji septembe...,Negatif
2460,Habis nganter super boss tutup rekening ini ba...,habis nganter super bos tutup rekening bank se...,Netral
1515,Aq dah pindah ke bcabentu solidarts saudaraku ...,deh pindah bcabentu solidarts saudara susah pa...,Negatif
3554,Kesian Mandiri kena paitnya birokrat 😅😅,kesi mandiri kena paitnya birokrat,Netral


## Train/test splits and shared helpers

In [7]:
import pandas as pd
import numpy as np

# Muat kembali data yang sudah bersih
df = pd.read_csv("clean_dataset.csv")

# Pastikan tidak ada data yang kosong (opsional tapi disarankan)
df = df.dropna(subset=['text_clean'])

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

assert df["label"].value_counts().min() >= 10, "A class has too few samples: refine the lexicon."

# BERT works best on natural text. Default = Steps 1-4 output (negations intact, no stemming).
# Set to "text_clean" to feed the fully stemmed text instead (strict PRD pipeline).
BERT_INPUT = "text_normalized"

# 80/20 split (Schema 1 & 2), stratified, same split reused for fair comparison
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=df["label"])

results = {}

def report(name, y_tr, p_tr, y_te, p_te):
    tr_acc, te_acc = accuracy_score(y_tr, p_tr), accuracy_score(y_te, p_te)
    results[name] = {"train_acc": tr_acc, "test_acc": te_acc}
    print(f"\n===== {name} =====")
    print(f"Train accuracy: {tr_acc:.4f} | Test accuracy: {te_acc:.4f}")
    print(classification_report(y_te, p_te, target_names=CLASS_NAMES, digits=4))

## Schema 1, IndoBERT + LoRA (dataset and model setup)

In [9]:
!pip install --upgrade torchao

In [18]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "(enable a GPU runtime in Colab for reasonable speed)")

BASE_MODEL  = "indobenchmark/indobert-base-p1"
BERT_OUT_DIR = "indobert_lora_sentiment"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)  # padding done per-batch
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

train_ds = SentimentDataset(df.loc[tr_idx, BERT_INPUT], df.loc[tr_idx, "label"])
test_ds  = SentimentDataset(df.loc[te_idx, BERT_INPUT], df.loc[te_idx, "label"])
collator = DataCollatorWithPadding(tokenizer)


ID2LABEL = {0: "Negatif", 1: "Netral", 2: "Positif"}
LABEL2ID = {"Negatif": 0, "Netral": 1, "Positif": 2}

# ---- Base model + LoRA adapters ----
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,        # also keeps the classifier head trainable
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["query", "value"], # BERT attention projections
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# ---- Class-imbalance handling: weighted cross-entropy ----
y_train = df.loc[tr_idx, "label"].values
cw = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float)
print("Class weights:", dict(zip(CLASS_NAMES, cw.round(3))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import f1_score
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

Device: cuda (enable a GPU runtime in Colab for reasonable speed)


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 592,131 || all params: 125,035,782 || trainable%: 0.4736
Class weights: {'Negatif': np.float64(1.196), 'Netral': np.float64(0.575), 'Positif': np.float64(2.348)}


Schema 1, training and evaluation

In [19]:
training_args = TrainingArguments(
    output_dir="./indobert_lora_ckpt",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,                  # LoRA tolerates (and needs) a higher LR than full fine-tuning
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    eval_strategy="epoch",               # use `evaluation_strategy` on transformers < 4.41
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    label_names=["labels"],
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ---- Final evaluation: train + test accuracy (PRD target > 92% on both) ----
train_pred = np.argmax(trainer.predict(train_ds).predictions, axis=-1)
test_pred  = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
report("Schema 1: IndoBERT + LoRA (80/20)",
       df.loc[tr_idx, "label"].values, train_pred,
       df.loc[te_idx, "label"].values, test_pred)

if min(results["Schema 1: IndoBERT + LoRA (80/20)"].values()) < 0.92:
    print("Below 92%: try more epochs, r=32, lr 3e-4, or audit/refine labels (see notes).")

# Persist adapter + tokenizer for the inference pipeline
trainer.save_model(BERT_OUT_DIR)
tokenizer.save_pretrained(BERT_OUT_DIR)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.969660,0.842030,0.596330,0.577936
2,0.721762,0.668421,0.745413,0.706773
3,0.499739,0.604436,0.717890,0.690105
4,0.438514,0.534445,0.779817,0.750585
5,0.318423,0.540227,0.777523,0.743877
6,0.265742,0.536059,0.794725,0.761149



===== Schema 1: IndoBERT + LoRA (80/20) =====
Train accuracy: 0.8950 | Test accuracy: 0.7947
              precision    recall  f1-score   support

     Negatif     0.7992    0.8189    0.8089       243
      Netral     0.8982    0.7861    0.8384       505
     Positif     0.5359    0.7823    0.6361       124

    accuracy                         0.7947       872
   macro avg     0.7444    0.7958    0.7611       872
weighted avg     0.8191    0.7947    0.8014       872

Below 92%: try more epochs, r=32, lr 3e-4, or audit/refine labels (see notes).


('indobert_lora_sentiment/tokenizer_config.json',
 'indobert_lora_sentiment/tokenizer.json')

## Schema 2, SVM + TF-IDF (80/20)

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import joblib

X_tr, X_te = df.loc[tr_idx, "text_clean"], df.loc[te_idx, "text_clean"]
y_tr, y_te = df.loc[tr_idx, "label"], df.loc[te_idx, "label"]

svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)),
    ("clf",   LinearSVC(C=1.0, class_weight="balanced", random_state=SEED)),  # linear-kernel SVM
])
svm_pipe.fit(X_tr, y_tr)
report("Schema 2: SVM + TF-IDF (80/20)", y_tr, svm_pipe.predict(X_tr), y_te, svm_pipe.predict(X_te))
joblib.dump(svm_pipe, "svm_tfidf.joblib")


===== Schema 2: SVM + TF-IDF (80/20) =====
Train accuracy: 0.9882 | Test accuracy: 0.8509
              precision    recall  f1-score   support

     Negatif     0.8686    0.8436    0.8559       243
      Netral     0.8911    0.8911    0.8911       505
     Positif     0.6641    0.7016    0.6824       124

    accuracy                         0.8509       872
   macro avg     0.8080    0.8121    0.8098       872
weighted avg     0.8526    0.8509    0.8516       872



['svm_tfidf.joblib']

## Schema 3, Random Forest + TF-IDF (70/30)

In [21]:
from sklearn.ensemble import RandomForestClassifier

tr3, te3 = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=df["label"])
X_tr3, X_te3 = df.loc[tr3, "text_clean"], df.loc[te3, "text_clean"]
y_tr3, y_te3 = df.loc[tr3, "label"], df.loc[te3, "label"]

rf_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)),
    ("clf",   RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                     n_jobs=-1, random_state=SEED)),
])
rf_pipe.fit(X_tr3, y_tr3)
report("Schema 3: Random Forest + TF-IDF (70/30)", y_tr3, rf_pipe.predict(X_tr3), y_te3, rf_pipe.predict(X_te3))
joblib.dump(rf_pipe, "rf_tfidf.joblib")


===== Schema 3: Random Forest + TF-IDF (70/30) =====
Train accuracy: 1.0000 | Test accuracy: 0.8417
              precision    recall  f1-score   support

     Negatif     0.8640    0.7857    0.8230       364
      Netral     0.8462    0.9367    0.8892       758
     Positif     0.7609    0.5645    0.6481       186

    accuracy                         0.8417      1308
   macro avg     0.8237    0.7623    0.7868      1308
weighted avg     0.8391    0.8417    0.8365      1308



['rf_tfidf.joblib']

Experiment summary

In [22]:
summary = pd.DataFrame(results).T
summary["target"] = [">92%", ">85%", ">85%"][:len(summary)]
summary[["train_acc", "test_acc"]] = summary[["train_acc", "test_acc"]].round(4)
summary

,train_acc,test_acc,target
Schema 1: IndoBERT + LoRA (80/20),0.8950,0.7947,>92%
Schema 2: SVM + TF-IDF (80/20),0.9882,0.8509,>85%
Schema 3: Random Forest + TF-IDF (70/30),1.0000,0.8417,>85%


In [28]:
!zip -r indobert_lora_ckpt.zip /content/indobert_lora_ckpt

  adding: content/indobert_lora_ckpt/ (stored 0%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/ (stored 0%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/scheduler.pt (deflated 62%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/trainer_state.json (deflated 75%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/adapter_config.json (deflated 60%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/tokenizer.json (deflated 71%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/optimizer.pt (deflated 8%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/tokenizer_config.json (deflated 43%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/adapter_model.safetensors (deflated 7%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/training_args.bin (deflated 53%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/scaler.pt (deflated 64%)
  adding: content/indobert_lora_ckpt/checkpoint-1308/rng_state.pth (deflated 26%)
  adding: content/indobert_lora_ckpt

In [29]:
!zip -r indobert_lora_sentiment.zip /content/indobert_lora_sentiment/


  adding: content/indobert_lora_sentiment/ (stored 0%)
  adding: content/indobert_lora_sentiment/adapter_config.json (deflated 60%)
  adding: content/indobert_lora_sentiment/tokenizer.json (deflated 71%)
  adding: content/indobert_lora_sentiment/tokenizer_config.json (deflated 43%)
  adding: content/indobert_lora_sentiment/adapter_model.safetensors (deflated 7%)
  adding: content/indobert_lora_sentiment/training_args.bin (deflated 53%)
  adding: content/indobert_lora_sentiment/README.md (deflated 66%)


Inference pipeline, predict_sentiment(raw_text)

In [23]:
from peft import PeftModel

# Reload the saved model from disk to prove the artifact works standalone
_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
infer_model = PeftModel.from_pretrained(_base, BERT_OUT_DIR).to(device).eval()
infer_tok   = AutoTokenizer.from_pretrained(BERT_OUT_DIR)


def predict_sentiment(raw_text: str) -> str:
    """Raw comment string -> preprocessing pipeline -> IndoBERT (LoRA) -> categorical label."""
    processed = full_pipeline(raw_text)[BERT_INPUT]
    if not processed.strip():
        return "Output: Netral (text is empty after preprocessing)"

    enc = infer_tok(processed, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = infer_model(**enc).logits
    pred_id = int(torch.argmax(logits, dim=-1).item())
    return f"Output: {ID2LABEL[pred_id]}"

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dummy test cases (the PRD’s “screenshot proof”)

In [30]:
import torch

# Pastikan model berjalan di perangkat yang tepat (GPU jika tersedia)
model.eval()

def predict_sentiment(text):
    # 1. Tokenisasi input teks baru
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding=True
    ).to(device) # Pindahkan ke GPU jika ada

    # 2. Lakukan prediksi tanpa menghitung gradien (agar cepat & hemat memori)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax(axis=-1).item()

    # 3. Kembalikan nama label dari ID2LABEL yang sudah kita buat sebelumnya
    predicted_label = ID2LABEL[predicted_class_id]
    return f"Predicted : {predicted_label}"

# --- Blok Testing Anda ---
test_cases = [
    ("POSITIF", "Saya setuju dengan langkah Bank Mandiri, sudah sesuai hukum dan prosedur. Tetap semangat!"),
    ("NETRAL",  "Ini kapan klarifikasi resmi dari pihak bank keluar ya?"),
    ("NEGATIF", "Kecewa banget, saya mau tarik semua dana dari Mandiri, komunikasinya arogan!!!"),
]

print("=" * 70)
for expected, text in test_cases:
    print(f"Expected : {expected}")
    print(f"Input    : {text}")
    print(predict_sentiment(text))
    print("-" * 70)

Expected : POSITIF
Input    : Saya setuju dengan langkah Bank Mandiri, sudah sesuai hukum dan prosedur. Tetap semangat!
Predicted : Positif
----------------------------------------------------------------------
Expected : NETRAL
Input    : Ini kapan klarifikasi resmi dari pihak bank keluar ya?
Predicted : Netral
----------------------------------------------------------------------
Expected : NEGATIF
Input    : Kecewa banget, saya mau tarik semua dana dari Mandiri, komunikasinya arogan!!!
Predicted : Negatif
----------------------------------------------------------------------


Generate requirements.txt with pipreqs

In [33]:
!pip install pipreqs nbconvert

# 1) Convert the notebook into a .py script
!jupyter nbconvert --to script mandiri_sentiment.ipynb

# 2) Generate requirements.txt from imports only (no pip-freeze bloat)
!pipreqs . --force --encoding utf-8

# 3) Add packages that are used implicitly
!echo "accelerate" >> requirements.txt
!cat requirements.txt

  Preparing metadata (setup.py) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=a791b0b795dea6fb3a4e4658d5a41382e2c905bdde968429154adc872231e35a
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
[NbConvertApp] WARNING | pattern 'mandiri_sentiment.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    